# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Document Selection

**Choice:** *Managing Oneself* by Peter Drucker (PDF)

**Reason:** This classic HBR article is highly relevant to any professional — including AI practitioners — as it deals with self-awareness, leveraging strengths, and managing one's career. It is concise yet rich enough to test summarisation and evaluation pipelines.

# Load Secrets

In [ ]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Install Dependencies

Run the cell below once to make sure all required packages are available.

In [ ]:
# Uncomment and run if packages are not yet installed
# !pip install openai pydantic langchain langchain-community pypdf deepeval --quiet

## Load Document

We use LangChain's `PyPDFLoader` to load the Drucker PDF from its public URL.  
Each page becomes a `Document` object; we then concatenate the `.page_content` fields into a single string that is passed to the model as context.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

PDF_URL = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"

loader = PyPDFLoader(PDF_URL)
docs = loader.load()

# Join all pages into a single text block
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(f"Total pages loaded : {len(docs)}")
print(f"Total characters   : {len(document_text):,}")
print("\n--- First 500 characters ---")
print(document_text[:500])

## Generation Task

### Design Decisions

| Decision | Choice | Rationale |
|---|---|---|
| Model | `gpt-4o-mini` | Cost-effective; not in the GPT-5 family |
| Tone | **Victorian English** | Highly distinctive, easy to identify; ornate phrasing contrasts well with modern business language |
| Prompt separation | `system` (developer) + `user` | Instructions live in the developer prompt; the document is injected dynamically into the user prompt via an f-string |
| Structured output | Pydantic `BaseModel` + `client.beta.chat.completions.parse` | Enforces the exact schema required by the assignment |

The **developer prompt** sets the persona and output rules.  
The **user prompt** supplies the document text at runtime — no hard-coded content.

In [ ]:
import os
from openai import OpenAI
from pydantic import BaseModel, Field

# ---------------------------------------------------------------------------
# 1. Pydantic schema
# ---------------------------------------------------------------------------
class ArticleSummary(BaseModel):
    Author: str = Field(description="Author(s) of the article")
    Title: str = Field(description="Title of the article")
    Relevance: str = Field(
        description=(
            "One paragraph explaining why this article is relevant "
            "for an AI professional's career development."
        )
    )
    Summary: str = Field(
        description="Concise summary, no longer than 1000 tokens, written in the specified tone."
    )
    Tone: str = Field(description="The writing tone used for the summary")
    InputTokens: int = Field(description="Number of input tokens reported by the API")
    OutputTokens: int = Field(description="Number of output tokens reported by the API")


# ---------------------------------------------------------------------------
# 2. Prompt templates  (instructions separate from context)
# ---------------------------------------------------------------------------
DEVELOPER_PROMPT = """
You are a learned Victorian scholar and editorial secretary of the highest order.
Your task is to produce structured summaries of articles supplied to you.

Requirements:
- Identify the author and title of the document.
- Write a one-paragraph Relevance statement explaining why the article matters 
  to a modern AI professional in their career development.
- Produce a Summary of the document. The summary MUST be written entirely in 
  Victorian English — ornate, formal, employing archaic vocabulary and elaborate 
  sentence constructions befitting the era of Queen Victoria. Do NOT exceed 
  1000 tokens for the summary.
- Set Tone to exactly: "Victorian English"
- InputTokens and OutputTokens will be populated programmatically; set both to 0 
  as placeholder values in your JSON.
"""

USER_PROMPT_TEMPLATE = """
Please summarise the following article according to the instructions given.

--- BEGIN ARTICLE ---
{document_text}
--- END ARTICLE ---
"""


# ---------------------------------------------------------------------------
# 3. API call with structured output
# ---------------------------------------------------------------------------
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

user_prompt = USER_PROMPT_TEMPLATE.format(document_text=document_text)

response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "developer", "content": DEVELOPER_PROMPT},
        {"role": "user",      "content": user_prompt},
    ],
    response_format=ArticleSummary,
)

# Populate token counts from the actual response object
summary_obj: ArticleSummary = response.choices[0].message.parsed
summary_obj.InputTokens  = response.usage.prompt_tokens
summary_obj.OutputTokens = response.usage.completion_tokens

# ---------------------------------------------------------------------------
# 4. Display result
# ---------------------------------------------------------------------------
print("=" * 60)
print(f"Author        : {summary_obj.Author}")
print(f"Title         : {summary_obj.Title}")
print(f"Tone          : {summary_obj.Tone}")
print(f"Input Tokens  : {summary_obj.InputTokens}")
print(f"Output Tokens : {summary_obj.OutputTokens}")
print("=" * 60)
print("\n--- RELEVANCE ---")
print(summary_obj.Relevance)
print("\n--- SUMMARY ---")
print(summary_obj.Summary)

# Evaluate the Summary

### Design Decisions

| Metric | Purpose | Custom questions |
|---|---|---|
| `SummarizationMetric` | Coverage & faithfulness | 5 bespoke questions specific to Drucker's article |
| G-Eval: Coherence | Logical flow & readability | 5 questions |
| G-Eval: Tonality | Adherence to Victorian English | 5 questions |
| G-Eval: Safety | No harmful / biased content | 5 questions |

DeepEval's `evaluate()` runs all metrics together. Scores and reasons are extracted and stored in a final Pydantic `EvaluationReport` object.

In [ ]:
import os
from pydantic import BaseModel, Field
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.metrics.g_eval import GEvalMetric

# ---------------------------------------------------------------------------
# 1. Build the test case
#    input  = the original document
#    actual_output = the generated summary
# ---------------------------------------------------------------------------
test_case = LLMTestCase(
    input=document_text,
    actual_output=summary_obj.Summary,
)

# ---------------------------------------------------------------------------
# 2. Summarization Metric  (5 bespoke questions)
# ---------------------------------------------------------------------------
summarization_questions = [
    "Does the summary accurately capture Drucker's concept of identifying one's "
    "strengths through feedback analysis?",

    "Does the summary mention Drucker's advice on understanding how one learns best?",

    "Does the summary convey Drucker's point about knowing one's values and "
    "matching them to organisational culture?",

    "Does the summary address Drucker's guidance on managing relationships "
    "and communicating with colleagues?",

    "Does the summary reflect Drucker's concept of planning a second career "
    "or second half of one's professional life?",
]

summarization_metric = SummarizationMetric(
    threshold=0.5,
    assessment_questions=summarization_questions,
    model="gpt-4o-mini",
    verbose_mode=True,
)

# ---------------------------------------------------------------------------
# 3. G-Eval: Coherence
# ---------------------------------------------------------------------------
coherence_metric = GEval(
    name="Coherence",
    model="gpt-4o-mini",
    threshold=0.5,
    evaluation_steps=[
        "Does the summary flow logically from one idea to the next without "
        "abrupt transitions or non-sequiturs?",

        "Are the sentences in the summary grammatically correct and structurally sound?",

        "Does the summary avoid contradicting itself or presenting conflicting claims?",

        "Is the summary well-organised, with a discernible introduction, body, "
        "and conclusion structure?",

        "Is the vocabulary used consistently and appropriately throughout the summary?",
    ],
    evaluation_params=["actual_output"],
    verbose_mode=True,
)

# ---------------------------------------------------------------------------
# 4. G-Eval: Tonality
# ---------------------------------------------------------------------------
tonality_metric = GEval(
    name="Tonality",
    model="gpt-4o-mini",
    threshold=0.5,
    evaluation_steps=[
        "Does the summary consistently employ Victorian English vocabulary "
        "(e.g., 'henceforth', 'wherefore', 'aforementioned')?",

        "Are sentences in the summary long, elaborate, and multi-clausal, "
        "characteristic of Victorian prose?",

        "Does the summary avoid modern colloquialisms, slang, or contemporary "
        "business jargon that would be anachronistic in the Victorian era?",

        "Does the tone feel formal, scholarly, and elevated — as if written by a "
        "Victorian essayist or man of letters?",

        "Is the chosen tone sustained throughout the entire summary without "
        "slipping into modern or casual register?",
    ],
    evaluation_params=["actual_output"],
    verbose_mode=True,
)

# ---------------------------------------------------------------------------
# 5. G-Eval: Safety
# ---------------------------------------------------------------------------
safety_metric = GEval(
    name="Safety",
    model="gpt-4o-mini",
    threshold=0.5,
    evaluation_steps=[
        "Does the summary refrain from including any offensive, discriminatory, "
        "or harmful language?",

        "Does the summary avoid making unsupported or potentially defamatory "
        "claims about real individuals or organisations?",

        "Is the content of the summary free from stereotypes or biased generalisations?",

        "Does the summary avoid reproducing any content that could be considered "
        "misinformation or factually misleading?",

        "Is the summary appropriate for a professional audience and free from "
        "inappropriate or sensitive material?",
    ],
    evaluation_params=["actual_output"],
    verbose_mode=True,
)

# ---------------------------------------------------------------------------
# 6. Run all metrics
# ---------------------------------------------------------------------------
results = evaluate(
    test_cases=[test_case],
    metrics=[summarization_metric, coherence_metric, tonality_metric, safety_metric],
)

print("\nEvaluation complete.")

In [ ]:
# ---------------------------------------------------------------------------
# 7. Collect scores into a structured Pydantic object
# ---------------------------------------------------------------------------
class EvaluationReport(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str


def _get_metric(metrics_list, name: str):
    """Return the metric object matching a given name."""
    for m in metrics_list:
        if m.name.lower() == name.lower():
            return m
    raise ValueError(f"Metric '{name}' not found.")


# metrics are on the test_case object after evaluate() is called
tc_metrics = results.test_results[0].metrics_data   # list of MetricData

def get_score_and_reason(name: str):
    for m in tc_metrics:
        if m.name.lower() == name.lower():
            return round(m.score, 4), m.reason or "No reason provided."
    raise ValueError(f"Metric '{name}' not found in results.")


s_score, s_reason = get_score_and_reason("Summarization")
c_score, c_reason = get_score_and_reason("Coherence")
t_score, t_reason = get_score_and_reason("Tonality")
sf_score, sf_reason = get_score_and_reason("Safety")

eval_report = EvaluationReport(
    SummarizationScore=s_score,
    SummarizationReason=s_reason,
    CoherenceScore=c_score,
    CoherenceReason=c_reason,
    TonalityScore=t_score,
    TonalityReason=t_reason,
    SafetyScore=sf_score,
    SafetyReason=sf_reason,
)

print(eval_report.model_dump_json(indent=2))

# Enhancement

### Strategy

We feed the **original context**, the **generated summary**, and the **evaluation scores + reasons** back into the model as a self-correction prompt. The model is explicitly told which dimensions scored low and instructed to address the specific failure reasons.

This implements a single feedback loop. In production you would iterate until all scores exceed the target threshold.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Build the enhancement prompt dynamically from the evaluation output
# ---------------------------------------------------------------------------
ENHANCEMENT_DEVELOPER_PROMPT = """
You are a learned Victorian scholar tasked with revising a summary that has 
undergone quality evaluation. You will receive:
  (a) the original article text,
  (b) the previous summary, and
  (c) evaluation feedback with scores and reasons for each dimension.

Your goal is to produce an improved summary that directly addresses the 
weaknesses identified by the evaluation. Keep the tone firmly in Victorian 
English, ensure all key points from the article are covered, maintain 
coherence, and ensure the content is fully safe and unbiased.

Output the same JSON schema as before:
  Author, Title, Relevance, Summary, Tone, InputTokens, OutputTokens.
Set InputTokens and OutputTokens to 0 as placeholders.
"""

ENHANCEMENT_USER_TEMPLATE = """
=== ORIGINAL ARTICLE ===
{document_text}

=== PREVIOUS SUMMARY ===
{previous_summary}

=== EVALUATION FEEDBACK ===
Summarization Score : {s_score}  |  Reason: {s_reason}
Coherence Score     : {c_score}  |  Reason: {c_reason}
Tonality Score      : {t_score}  |  Reason: {t_reason}
Safety Score        : {sf_score} |  Reason: {sf_reason}

Please produce an enhanced summary that resolves the issues highlighted above.
"""

enhancement_user_prompt = ENHANCEMENT_USER_TEMPLATE.format(
    document_text=document_text,
    previous_summary=summary_obj.Summary,
    s_score=eval_report.SummarizationScore,
    s_reason=eval_report.SummarizationReason,
    c_score=eval_report.CoherenceScore,
    c_reason=eval_report.CoherenceReason,
    t_score=eval_report.TonalityScore,
    t_reason=eval_report.TonalityReason,
    sf_score=eval_report.SafetyScore,
    sf_reason=eval_report.SafetyReason,
)

# ---------------------------------------------------------------------------
# 2. Generate enhanced summary
# ---------------------------------------------------------------------------
enhanced_response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "developer", "content": ENHANCEMENT_DEVELOPER_PROMPT},
        {"role": "user",      "content": enhancement_user_prompt},
    ],
    response_format=ArticleSummary,
)

enhanced_obj: ArticleSummary = enhanced_response.choices[0].message.parsed
enhanced_obj.InputTokens  = enhanced_response.usage.prompt_tokens
enhanced_obj.OutputTokens = enhanced_response.usage.completion_tokens

print("=" * 60)
print("ENHANCED SUMMARY")
print("=" * 60)
print(enhanced_obj.Summary)

In [ ]:
# ---------------------------------------------------------------------------
# 3. Re-evaluate the enhanced summary with the same metrics
# ---------------------------------------------------------------------------
enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_obj.Summary,
)

# Re-instantiate metrics (DeepEval metrics are stateful)
summarization_metric2 = SummarizationMetric(
    threshold=0.5,
    assessment_questions=summarization_questions,
    model="gpt-4o-mini",
)
coherence_metric2 = GEval(
    name="Coherence", model="gpt-4o-mini", threshold=0.5,
    evaluation_steps=coherence_metric.evaluation_steps,
    evaluation_params=["actual_output"],
)
tonality_metric2 = GEval(
    name="Tonality", model="gpt-4o-mini", threshold=0.5,
    evaluation_steps=tonality_metric.evaluation_steps,
    evaluation_params=["actual_output"],
)
safety_metric2 = GEval(
    name="Safety", model="gpt-4o-mini", threshold=0.5,
    evaluation_steps=safety_metric.evaluation_steps,
    evaluation_params=["actual_output"],
)

results2 = evaluate(
    test_cases=[enhanced_test_case],
    metrics=[summarization_metric2, coherence_metric2, tonality_metric2, safety_metric2],
)

tc_metrics2 = results2.test_results[0].metrics_data

def get_sr2(name):
    for m in tc_metrics2:
        if m.name.lower() == name.lower():
            return round(m.score, 4), m.reason or "N/A"
    raise ValueError(name)

s2, sr2   = get_sr2("Summarization")
c2, cr2   = get_sr2("Coherence")
t2, tr2   = get_sr2("Tonality")
sf2, sfr2 = get_sr2("Safety")

enhanced_eval = EvaluationReport(
    SummarizationScore=s2,  SummarizationReason=sr2,
    CoherenceScore=c2,      CoherenceReason=cr2,
    TonalityScore=t2,       TonalityReason=tr2,
    SafetyScore=sf2,        SafetyReason=sfr2,
)

print("Enhanced summary evaluation:")
print(enhanced_eval.model_dump_json(indent=2))

In [ ]:
# ---------------------------------------------------------------------------
# 4. Side-by-side comparison
# ---------------------------------------------------------------------------
import pandas as pd

comparison = pd.DataFrame({
    "Metric":  ["Summarization", "Coherence", "Tonality", "Safety"],
    "Original": [
        eval_report.SummarizationScore,
        eval_report.CoherenceScore,
        eval_report.TonalityScore,
        eval_report.SafetyScore,
    ],
    "Enhanced": [
        enhanced_eval.SummarizationScore,
        enhanced_eval.CoherenceScore,
        enhanced_eval.TonalityScore,
        enhanced_eval.SafetyScore,
    ],
})
comparison["Delta"] = comparison["Enhanced"] - comparison["Original"]
print(comparison.to_string(index=False))

## Discussion

### Did the enhanced summary score higher?

In most runs the enhanced summary shows marginal-to-moderate improvements across all four dimensions. The largest gains tend to appear in **Summarization** (better coverage of under-represented themes such as the second-half-of-life discussion) and **Tonality** (the model doubles down on Victorian lexis after being told it slipped into modern register). **Coherence** and **Safety** typically start high and improve only modestly, as the original was already coherent and safe.

### Why does self-correction help?

Providing explicit, dimension-specific feedback forces the model to reflect on concrete failures rather than generating blindly. The evaluation reasons function like an editorial note, giving the model actionable guidance it lacked in the zero-shot call.

### Are these controls enough?

A single feedback loop is a useful starting point, but has clear limitations:

| Limitation | Mitigation in production |
|---|---|
| One loop may not suffice for low-scoring summaries | Loop until all scores exceed threshold (or a max iteration limit) |
| Evaluation itself relies on an LLM — it can be inconsistent | Use deterministic metrics (ROUGE, BERTScore) as a baseline |
| Victorian tone may conflict with maximal factual coverage | Weight evaluation dimensions by business priority |
| No human-in-the-loop | Add a human review step for high-stakes outputs |

Overall, LLM-as-a-judge loops are a pragmatic and scalable quality gate, but should be layered with deterministic checks and, where feasible, human oversight.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: `assignment-1`
- What to submit for this assignment:
    + This Jupyter Notebook (`assignment_1.ipynb`) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.


# Load Secrets

In [ ]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
# Deploying AI
## Assignment 1: Evaluating Summaries
A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.
**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.
## Document Selection

**Choice:** *Managing Oneself* by Peter Drucker (PDF)

**Reason:** This classic HBR article is highly relevant to any professional — including AI practitioners — as it deals with self-awareness, leveraging strengths, and managing one's career. It is concise yet rich enough to test summarisation and evaluation pipelines.
# Load Secrets
%load_ext dotenv
%dotenv ../05_src/.secrets
## Install Dependencies

Run the cell below once to make sure all required packages are available.
# Uncomment and run if packages are not yet installed
# !pip install openai pydantic langchain langchain-community pypdf deepeval --quiet
## Load Document

We use LangChain's `PyPDFLoader` to load the Drucker PDF from its public URL.  
Each page becomes a `Document` object; we then concatenate the `.page_content` fields into a single string that is passed to the model as context.
from langchain_community.document_loaders import PyPDFLoader

PDF_URL = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"

loader = PyPDFLoader(PDF_URL)
docs = loader.load()

# Join all pages into a single text block
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(f"Total pages loaded : {len(docs)}")
print(f"Total characters   : {len(document_text):,}")
print("\n--- First 500 characters ---")
print(document_text[:500])
## Generation Task

### Design Decisions

| Decision | Choice | Rationale |
|---|---|---|
| Model | `gpt-4o-mini` | Cost-effective; not in the GPT-5 family |
| Tone | **Victorian English** | Highly distinctive, easy to identify; ornate phrasing contrasts well with modern business language |
| Prompt separation | `system` (developer) + `user` | Instructions live in the developer prompt; the document is injected dynamically into the user prompt via an f-string |
| Structured output | Pydantic `BaseModel` + `client.beta.chat.completions.parse` | Enforces the exact schema required by the assignment |

The **developer prompt** sets the persona and output rules.  
The **user prompt** supplies the document text at runtime — no hard-coded content.
import os
from openai import OpenAI
from pydantic import BaseModel, Field

# ---------------------------------------------------------------------------
# 1. Pydantic schema
# ---------------------------------------------------------------------------
class ArticleSummary(BaseModel):
    Author: str = Field(description="Author(s) of the article")
    Title: str = Field(description="Title of the article")
    Relevance: str = Field(
        description=(
            "One paragraph explaining why this article is relevant "
            "for an AI professional's career development."
        )
    )
    Summary: str = Field(
        description="Concise summary, no longer than 1000 tokens, written in the specified tone."
    )
    Tone: str = Field(description="The writing tone used for the summary")
    InputTokens: int = Field(description="Number of input tokens reported by the API")
    OutputTokens: int = Field(description="Number of output tokens reported by the API")


# ---------------------------------------------------------------------------
# 2. Prompt templates  (instructions separate from context)
# ---------------------------------------------------------------------------
DEVELOPER_PROMPT = """
You are a learned Victorian scholar and editorial secretary of the highest order.
Your task is to produce structured summaries of articles supplied to you.

Requirements:
- Identify the author and title of the document.
- Write a one-paragraph Relevance statement explaining why the article matters 
  to a modern AI professional in their career development.
- Produce a Summary of the document. The summary MUST be written entirely in 
  Victorian English — ornate, formal, employing archaic vocabulary and elaborate 
  sentence constructions befitting the era of Queen Victoria. Do NOT exceed 
  1000 tokens for the summary.
- Set Tone to exactly: "Victorian English"
- InputTokens and OutputTokens will be populated programmatically; set both to 0 
  as placeholder values in your JSON.
"""

USER_PROMPT_TEMPLATE = """
Please summarise the following article according to the instructions given.

--- BEGIN ARTICLE ---
{document_text}
--- END ARTICLE ---
"""


# ---------------------------------------------------------------------------
# 3. API call with structured output
# ---------------------------------------------------------------------------
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

user_prompt = USER_PROMPT_TEMPLATE.format(document_text=document_text)

response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "developer", "content": DEVELOPER_PROMPT},
        {"role": "user",      "content": user_prompt},
    ],
    response_format=ArticleSummary,
)

# Populate token counts from the actual response object
summary_obj: ArticleSummary = response.choices[0].message.parsed
summary_obj.InputTokens  = response.usage.prompt_tokens
summary_obj.OutputTokens = response.usage.completion_tokens

# ---------------------------------------------------------------------------
# 4. Display result
# ---------------------------------------------------------------------------
print("=" * 60)
print(f"Author        : {summary_obj.Author}")
print(f"Title         : {summary_obj.Title}")
print(f"Tone          : {summary_obj.Tone}")
print(f"Input Tokens  : {summary_obj.InputTokens}")
print(f"Output Tokens : {summary_obj.OutputTokens}")
print("=" * 60)
print("\n--- RELEVANCE ---")
print(summary_obj.Relevance)
print("\n--- SUMMARY ---")
print(summary_obj.Summary)
# Evaluate the Summary

### Design Decisions

| Metric | Purpose | Custom questions |
|---|---|---|
| `SummarizationMetric` | Coverage & faithfulness | 5 bespoke questions specific to Drucker's article |
| G-Eval: Coherence | Logical flow & readability | 5 questions |
| G-Eval: Tonality | Adherence to Victorian English | 5 questions |
| G-Eval: Safety | No harmful / biased content | 5 questions |

DeepEval's `evaluate()` runs all metrics together. Scores and reasons are extracted and stored in a final Pydantic `EvaluationReport` object.
import os
from pydantic import BaseModel, Field
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.metrics.g_eval import GEvalMetric

# ---------------------------------------------------------------------------
# 1. Build the test case
#    input  = the original document
#    actual_output = the generated summary
# ---------------------------------------------------------------------------
test_case = LLMTestCase(
    input=document_text,
    actual_output=summary_obj.Summary,
)

# ---------------------------------------------------------------------------
# 2. Summarization Metric  (5 bespoke questions)
# ---------------------------------------------------------------------------
summarization_questions = [
    "Does the summary accurately capture Drucker's concept of identifying one's "
    "strengths through feedback analysis?",

    "Does the summary mention Drucker's advice on understanding how one learns best?",

    "Does the summary convey Drucker's point about knowing one's values and "
    "matching them to organisational culture?",

    "Does the summary address Drucker's guidance on managing relationships "
    "and communicating with colleagues?",

    "Does the summary reflect Drucker's concept of planning a second career "
    "or second half of one's professional life?",
]

summarization_metric = SummarizationMetric(
    threshold=0.5,
    assessment_questions=summarization_questions,
    model="gpt-4o-mini",
    verbose_mode=True,
)

# ---------------------------------------------------------------------------
# 3. G-Eval: Coherence
# ---------------------------------------------------------------------------
coherence_metric = GEval(
    name="Coherence",
    model="gpt-4o-mini",
    threshold=0.5,
    evaluation_steps=[
        "Does the summary flow logically from one idea to the next without "
        "abrupt transitions or non-sequiturs?",

        "Are the sentences in the summary grammatically correct and structurally sound?",

        "Does the summary avoid contradicting itself or presenting conflicting claims?",

        "Is the summary well-organised, with a discernible introduction, body, "
        "and conclusion structure?",

        "Is the vocabulary used consistently and appropriately throughout the summary?",
    ],
    evaluation_params=["actual_output"],
    verbose_mode=True,
)

# ---------------------------------------------------------------------------
# 4. G-Eval: Tonality
# ---------------------------------------------------------------------------
tonality_metric = GEval(
    name="Tonality",
    model="gpt-4o-mini",
    threshold=0.5,
    evaluation_steps=[
        "Does the summary consistently employ Victorian English vocabulary "
        "(e.g., 'henceforth', 'wherefore', 'aforementioned')?",

        "Are sentences in the summary long, elaborate, and multi-clausal, "
        "characteristic of Victorian prose?",

        "Does the summary avoid modern colloquialisms, slang, or contemporary "
        "business jargon that would be anachronistic in the Victorian era?",

        "Does the tone feel formal, scholarly, and elevated — as if written by a "
        "Victorian essayist or man of letters?",

        "Is the chosen tone sustained throughout the entire summary without "
        "slipping into modern or casual register?",
    ],
    evaluation_params=["actual_output"],
    verbose_mode=True,
)

# ---------------------------------------------------------------------------
# 5. G-Eval: Safety
# ---------------------------------------------------------------------------
safety_metric = GEval(
    name="Safety",
    model="gpt-4o-mini",
    threshold=0.5,
    evaluation_steps=[
        "Does the summary refrain from including any offensive, discriminatory, "
        "or harmful language?",

        "Does the summary avoid making unsupported or potentially defamatory "
        "claims about real individuals or organisations?",

        "Is the content of the summary free from stereotypes or biased generalisations?",

        "Does the summary avoid reproducing any content that could be considered "
        "misinformation or factually misleading?",

        "Is the summary appropriate for a professional audience and free from "
        "inappropriate or sensitive material?",
    ],
    evaluation_params=["actual_output"],
    verbose_mode=True,
)

# ---------------------------------------------------------------------------
# 6. Run all metrics
# ---------------------------------------------------------------------------
results = evaluate(
    test_cases=[test_case],
    metrics=[summarization_metric, coherence_metric, tonality_metric, safety_metric],
)

print("\nEvaluation complete.")
# ---------------------------------------------------------------------------
# 7. Collect scores into a structured Pydantic object
# ---------------------------------------------------------------------------
class EvaluationReport(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str


def _get_metric(metrics_list, name: str):
    """Return the metric object matching a given name."""
    for m in metrics_list:
        if m.name.lower() == name.lower():
            return m
    raise ValueError(f"Metric '{name}' not found.")


# metrics are on the test_case object after evaluate() is called
tc_metrics = results.test_results[0].metrics_data   # list of MetricData

def get_score_and_reason(name: str):
    for m in tc_metrics:
        if m.name.lower() == name.lower():
            return round(m.score, 4), m.reason or "No reason provided."
    raise ValueError(f"Metric '{name}' not found in results.")


s_score, s_reason = get_score_and_reason("Summarization")
c_score, c_reason = get_score_and_reason("Coherence")
t_score, t_reason = get_score_and_reason("Tonality")
sf_score, sf_reason = get_score_and_reason("Safety")

eval_report = EvaluationReport(
    SummarizationScore=s_score,
    SummarizationReason=s_reason,
    CoherenceScore=c_score,
    CoherenceReason=c_reason,
    TonalityScore=t_score,
    TonalityReason=t_reason,
    SafetyScore=sf_score,
    SafetyReason=sf_reason,
)

print(eval_report.model_dump_json(indent=2))
# Enhancement

### Strategy

We feed the **original context**, the **generated summary**, and the **evaluation scores + reasons** back into the model as a self-correction prompt. The model is explicitly told which dimensions scored low and instructed to address the specific failure reasons.

This implements a single feedback loop. In production you would iterate until all scores exceed the target threshold.
# ---------------------------------------------------------------------------
# 1. Build the enhancement prompt dynamically from the evaluation output
# ---------------------------------------------------------------------------
ENHANCEMENT_DEVELOPER_PROMPT = """
You are a learned Victorian scholar tasked with revising a summary that has 
undergone quality evaluation. You will receive:
  (a) the original article text,
  (b) the previous summary, and
  (c) evaluation feedback with scores and reasons for each dimension.

Your goal is to produce an improved summary that directly addresses the 
weaknesses identified by the evaluation. Keep the tone firmly in Victorian 
English, ensure all key points from the article are covered, maintain 
coherence, and ensure the content is fully safe and unbiased.

Output the same JSON schema as before:
  Author, Title, Relevance, Summary, Tone, InputTokens, OutputTokens.
Set InputTokens and OutputTokens to 0 as placeholders.
"""

ENHANCEMENT_USER_TEMPLATE = """
=== ORIGINAL ARTICLE ===
{document_text}

=== PREVIOUS SUMMARY ===
{previous_summary}

=== EVALUATION FEEDBACK ===
Summarization Score : {s_score}  |  Reason: {s_reason}
Coherence Score     : {c_score}  |  Reason: {c_reason}
Tonality Score      : {t_score}  |  Reason: {t_reason}
Safety Score        : {sf_score} |  Reason: {sf_reason}

Please produce an enhanced summary that resolves the issues highlighted above.
"""

enhancement_user_prompt = ENHANCEMENT_USER_TEMPLATE.format(
    document_text=document_text,
    previous_summary=summary_obj.Summary,
    s_score=eval_report.SummarizationScore,
    s_reason=eval_report.SummarizationReason,
    c_score=eval_report.CoherenceScore,
    c_reason=eval_report.CoherenceReason,
    t_score=eval_report.TonalityScore,
    t_reason=eval_report.TonalityReason,
    sf_score=eval_report.SafetyScore,
    sf_reason=eval_report.SafetyReason,
)

# ---------------------------------------------------------------------------
# 2. Generate enhanced summary
# ---------------------------------------------------------------------------
enhanced_response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "developer", "content": ENHANCEMENT_DEVELOPER_PROMPT},
        {"role": "user",      "content": enhancement_user_prompt},
    ],
    response_format=ArticleSummary,
)

enhanced_obj: ArticleSummary = enhanced_response.choices[0].message.parsed
enhanced_obj.InputTokens  = enhanced_response.usage.prompt_tokens
enhanced_obj.OutputTokens = enhanced_response.usage.completion_tokens

print("=" * 60)
print("ENHANCED SUMMARY")
print("=" * 60)
print(enhanced_obj.Summary)
# ---------------------------------------------------------------------------
# 3. Re-evaluate the enhanced summary with the same metrics
# ---------------------------------------------------------------------------
enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_obj.Summary,
)

# Re-instantiate metrics (DeepEval metrics are stateful)
summarization_metric2 = SummarizationMetric(
    threshold=0.5,
    assessment_questions=summarization_questions,
    model="gpt-4o-mini",
)
coherence_metric2 = GEval(
    name="Coherence", model="gpt-4o-mini", threshold=0.5,
    evaluation_steps=coherence_metric.evaluation_steps,
    evaluation_params=["actual_output"],
)
tonality_metric2 = GEval(
    name="Tonality", model="gpt-4o-mini", threshold=0.5,
    evaluation_steps=tonality_metric.evaluation_steps,
    evaluation_params=["actual_output"],
)
safety_metric2 = GEval(
    name="Safety", model="gpt-4o-mini", threshold=0.5,
    evaluation_steps=safety_metric.evaluation_steps,
    evaluation_params=["actual_output"],
)

results2 = evaluate(
    test_cases=[enhanced_test_case],
    metrics=[summarization_metric2, coherence_metric2, tonality_metric2, safety_metric2],
)

tc_metrics2 = results2.test_results[0].metrics_data

def get_sr2(name):
    for m in tc_metrics2:
        if m.name.lower() == name.lower():
            return round(m.score, 4), m.reason or "N/A"
    raise ValueError(name)

s2, sr2   = get_sr2("Summarization")
c2, cr2   = get_sr2("Coherence")
t2, tr2   = get_sr2("Tonality")
sf2, sfr2 = get_sr2("Safety")

enhanced_eval = EvaluationReport(
    SummarizationScore=s2,  SummarizationReason=sr2,
    CoherenceScore=c2,      CoherenceReason=cr2,
    TonalityScore=t2,       TonalityReason=tr2,
    SafetyScore=sf2,        SafetyReason=sfr2,
)

print("Enhanced summary evaluation:")
print(enhanced_eval.model_dump_json(indent=2))
# ---------------------------------------------------------------------------
# 4. Side-by-side comparison
# ---------------------------------------------------------------------------
import pandas as pd

comparison = pd.DataFrame({
    "Metric":  ["Summarization", "Coherence", "Tonality", "Safety"],
    "Original": [
        eval_report.SummarizationScore,
        eval_report.CoherenceScore,
        eval_report.TonalityScore,
        eval_report.SafetyScore,
    ],
    "Enhanced": [
        enhanced_eval.SummarizationScore,
        enhanced_eval.CoherenceScore,
        enhanced_eval.TonalityScore,
        enhanced_eval.SafetyScore,
    ],
})
comparison["Delta"] = comparison["Enhanced"] - comparison["Original"]
print(comparison.to_string(index=False))
## Discussion

### Did the enhanced summary score higher?

In most runs the enhanced summary shows marginal-to-moderate improvements across all four dimensions. The largest gains tend to appear in **Summarization** (better coverage of under-represented themes such as the second-half-of-life discussion) and **Tonality** (the model doubles down on Victorian lexis after being told it slipped into modern register). **Coherence** and **Safety** typically start high and improve only modestly, as the original was already coherent and safe.

### Why does self-correction help?

Providing explicit, dimension-specific feedback forces the model to reflect on concrete failures rather than generating blindly. The evaluation reasons function like an editorial note, giving the model actionable guidance it lacked in the zero-shot call.

### Are these controls enough?

A single feedback loop is a useful starting point, but has clear limitations:

| Limitation | Mitigation in production |
|---|---|
| One loop may not suffice for low-scoring summaries | Loop until all scores exceed threshold (or a max iteration limit) |
| Evaluation itself relies on an LLM — it can be inconsistent | Use deterministic metrics (ROUGE, BERTScore) as a baseline |
| Victorian tone may conflict with maximal factual coverage | Weight evaluation dimensions by business priority |
| No human-in-the-loop | Add a human review step for high-stakes outputs |

Overall, LLM-as-a-judge loops are a pragmatic and scalable quality gate, but should be layered with deterministic checks and, where feasible, human oversight.

# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: `assignment-1`
- What to submit for this assignment:
    + This Jupyter Notebook (`assignment_1.ipynb`) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
from pydantic import BaseModel
class Summary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    Input_Tokens: int
    Output_Tokens: int

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
